# DORA RAG — Phase 4 (v2): Retrieval Pipeline

**Fix from v1:** `QueryFusionRetriever` internally calls `Settings.llm` even
when `num_queries=1`, causing LlamaIndex to fall back to OpenAI and fail.
This version explicitly sets `Settings.llm` to a mock/Groq before any
retriever is constructed, preventing the OpenAI lookup entirely.

**Prerequisites:** Phases 1, 2, 3 complete. `dora_rag/index_config.json` must exist.

In [1]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-index",
    "llama-index-llms-groq",
    "llama-index-embeddings-huggingface",
    "llama-index-vector-stores-chroma",
    "llama-index-retrievers-bm25",
    "chromadb",
    "sentence-transformers",
    "torch",
    "rank-bm25",
    "PyStemmer",
])
print("Dependencies installed.")


Dependencies installed.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
import os
# Cell 2 — Imports, config, LLM setup
import json, time, re as _re, datetime, torch
from pathlib import Path
from collections import Counter
from llama_index.core import Settings
from llama_index.llms.groq import Groq

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')
router_llm = Groq(model='llama-3.1-8b-instant', api_key=GROQ_API_KEY)
Settings.llm = router_llm
print('✓ Settings.llm set to Groq 8b')
llm = Groq(model='llama-3.3-70b-versatile', api_key=GROQ_API_KEY)
print('✓ Generation LLM: Groq llama-3.3-70b-versatile')

BASE_DIR    = Path('./dora_rag')
CHUNKS_DIR  = BASE_DIR / 'dora_chunks'
# Phase 4b reads all_chunks.json (all 3 regulations, 798 chunks)
# Phase 4a (DORA-only) continues to use dora_chunks.json unchanged
CHUNKS_FILE = CHUNKS_DIR / 'all_chunks.json'
CONFIG_FILE = BASE_DIR / 'index_config.json'

if not CONFIG_FILE.exists():
    raise FileNotFoundError('index_config.json not found. Run 03b first.')

config = json.loads(CONFIG_FILE.read_text())
print(f'\nIndex config loaded:')
print(f'  Model        : {config["embed_model"]}')
print(f'  Collection   : {config["chroma_collection"]}')
print(f'  Total vectors: {config["total_vectors"]}')
print(f'  Regulations  : {config.get("corpus_regulations", {})}')
print(f'  Levels       : {config.get("corpus_levels", {})}')

CHROMA_DIR        = Path(config['chroma_dir'])
CHROMA_COLLECTION = config['chroma_collection']
EMBED_MODEL_NAME  = config['embed_model']
device            = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'  Device       : {device}')


✓ Settings.llm set to Groq 8b — OpenAI fallback disabled
✓ Generation LLM: Groq llama-3.3-70b-versatile

Index config loaded:
  Model        : BAAI/bge-large-en-v1.5
  Collection   : dora_regulation
  Total vectors: 798
  Regulations  : {'dora': 381, 'nis2': 179, 'ps21_3': 119, 'cross_reg': 119}
  Levels       : {'1': 276, '2': 440, '3': 82}
  Device       : cpu


In [4]:
# Cell 3 — Load embedding model
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print(f"Loading {EMBED_MODEL_NAME} on {device}...")
t0 = time.time()

embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    query_instruction="Represent this sentence for searching relevant passages: ",
    embed_batch_size=16,
    device=device,
)
Settings.embed_model = embed_model

print(f"✓ Model loaded in {time.time()-t0:.1f}s")

Loading BAAI/bge-large-en-v1.5 on cpu...
✓ Model loaded in 2.0s


In [5]:
# Cell 4 — Load Chroma index
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

chroma_client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
chroma_collection = chroma_client.get_collection(CHROMA_COLLECTION)
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context   = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)

print(f"✓ Index loaded — {chroma_collection.count()} vectors")

✓ Index loaded — 798 vectors


In [6]:
# Cell 5 — Load all_chunks.json and build TextNodes for BM25
from llama_index.core.schema import TextNode

if not CHUNKS_FILE.exists():
    raise FileNotFoundError(f'Chunks file not found: {CHUNKS_FILE}')

raw_chunks = json.loads(CHUNKS_FILE.read_text(encoding='utf-8'))

all_nodes = [
    TextNode(
        id_  = c['chunk_id'],
        text = c['text_with_header'],
        metadata = {
            'regulation':    c.get('regulation', 'dora'),
            'jurisdiction':  c.get('jurisdiction', 'eu'),
            'source_doc':    c['source_doc'],
            'doc_type':      c['doc_type'],
            'level':         c.get('level', 0),
            'article_num':   c['article_num'] if c['article_num'] is not None else -1,
            'article_title': c['article_title'],
            'section_ref':   c.get('section_ref') or '',
            'paragraph_idx': c['paragraph_idx'],
            'topic_tags':    '|'.join(c['topic_tags']),
            'entity_scope':  '|'.join(c['entity_scope']),
            'token_estimate':c['token_estimate'],
        }
    ) for c in raw_chunks
]

print(f'✓ Loaded {len(all_nodes)} nodes')
by_reg = Counter(c.get('regulation','?') for c in raw_chunks)
print('\nChunks by regulation:')
for reg, cnt in sorted(by_reg.items()):
    print(f'  {reg:<14}: {cnt}')
by_source = Counter(c['source_doc'] for c in raw_chunks)
print('\nChunks per document:')
for src, count in sorted(by_source.items()):
    print(f'  {src[:52]:<52} {count:>5}')


✓ Loaded 826 nodes

Chunks by regulation:
  cross_reg     : 128
  dora          : 390
  nis2          : 189
  ps21_3        : 119

Chunks per document:
  DORA_ESA_Guidelines_Simplified_ICT_Framework            18
  DORA_ESA_QA_Implementation_Guidance                     43
  DP3_22_Critical_Third_Parties_UK                        57
  EC_Guidelines_DORA_NIS2_Relationship_2023                3
  FCA_FG22_5_Operational_Resilience                       25
  L1_DORA_Directive_2022_2556                             16
  L1_DORA_Regulation_2022_2554                           116
  L2_DR_CTPP_Designation_Criteria                          7
  L2_DR_DORA_Oversight_Fees                                7
  L2_ITS_ICT_Incident_Reporting                           47
  L2_ITS_Register_of_Information                          47
  L2_RTS_ICT_Incident_Classification                      14
  L2_RTS_ICT_Incident_Reporting_Process                    8
  L2_RTS_ICT_Risk_Management                           

In [7]:
# Cell 6 — Build BM25 retriever
from llama_index.retrievers.bm25 import BM25Retriever

try:
    import Stemmer
    stemmer = Stemmer.Stemmer("english")
    print("Using PyStemmer for BM25")
except ImportError:
    stemmer = None
    print("PyStemmer not found — BM25 will run without stemming (still works)")

bm25_retriever = BM25Retriever.from_defaults(
    nodes=all_nodes,
    similarity_top_k=5,
    stemmer=stemmer,
    language="english",
)

print(f"✓ BM25 retriever built over {len(all_nodes)} nodes")

Using PyStemmer for BM25
✓ BM25 retriever built over 826 nodes


In [8]:
# Cell 7 — Build semantic retriever
from llama_index.core.retrievers import VectorIndexRetriever

semantic_retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5,
)

print("✓ Semantic retriever ready")

✓ Semantic retriever ready


In [11]:
# Cell 8 — Build hybrid retriever
#
# Now safe because Settings.llm was set to Groq in Cell 2.
# QueryFusionRetriever will use Groq if it needs an LLM internally,
# but with num_queries=1 it performs pure RRF fusion without any LLM calls.

from llama_index.core.retrievers import QueryFusionRetriever

hybrid_retriever = QueryFusionRetriever(
    retrievers=[semantic_retriever, bm25_retriever],
    llm=Settings.llm,            # explicitly pass Groq — no ambiguity
    similarity_top_k=5,
    num_queries=1,               # no query expansion — pure RRF fusion
    mode="reciprocal_rerank",
    use_async=False,
    verbose=False,
)

print("✓ Hybrid retriever ready (semantic + BM25, RRF fusion)")

✓ Hybrid retriever ready (semantic + BM25, RRF fusion)


In [12]:
# Cell 9 — Display helper
def show_results(results, label: str = ""):
    if label:
        print(f"\n{'═'*65}")
        print(f"  {label}")
        print(f"{'═'*65}")
    print(f"  Results : {len(results)}")
    print(f"  {'#':<3} {'L':>2} {'Type':<22} {'Art':>4}  {'Score':>6}  Source / Title")
    print(f"  {'─'*62}")
    for i, node in enumerate(results, 1):
        m     = node.metadata
        lv    = m.get('level', '?')
        dtype = m.get('doc_type', '?')
        art   = m.get('article_num', '?')
        title = m.get('article_title', '')[:28]
        src   = m.get('source_doc', '')[:28]
        score = getattr(node, 'score', 0.0) or 0.0
        print(f"  {i:<3} {lv:>2} {dtype:<22} {str(art):>4}  {score:>6.4f}  {src} — {title}")
    print()

print("Display helper ready.")

Display helper ready.


In [13]:
# Cell 10 — Compare: semantic vs BM25 vs hybrid on same query
TEST_QUERY = "What are the reporting timelines for major ICT incidents?"
print(f"Query: {TEST_QUERY}\n")

sem_results    = semantic_retriever.retrieve(TEST_QUERY)
bm25_results   = bm25_retriever.retrieve(TEST_QUERY)
hybrid_results = hybrid_retriever.retrieve(TEST_QUERY)

show_results(sem_results,    "SEMANTIC RETRIEVER")
show_results(bm25_results,   "BM25 RETRIEVER")
show_results(hybrid_results, "HYBRID RETRIEVER (recommended)")

sem_ids  = {n.id_ for n in sem_results}
bm25_ids = {n.id_ for n in bm25_results}
print(f"Overlap (semantic ∩ BM25) : {len(sem_ids & bm25_ids)} chunk(s)")
print(f"Unique to semantic        : {len(sem_ids - bm25_ids)}")
print(f"Unique to BM25            : {len(bm25_ids - sem_ids)}")

Query: What are the reporting timelines for major ICT incidents?


═════════════════════════════════════════════════════════════════
  SEMANTIC RETRIEVER
═════════════════════════════════════════════════════════════════
  Results : 5
  #    L Type                    Art   Score  Source / Title
  ──────────────────────────────────────────────────────────────
  1    1 regulation               21  0.7449  L1_DORA_Regulation_2022_2554 — Centralisation of reporting 
  2    2 ITS                       9  0.7447  L2_ITS_ICT_Incident_Reportin — Entry into force
  3    2 ITS                       5  0.7413  L2_ITS_ICT_Incident_Reportin — Reclassification of major IC
  4    2 ITS                       9  0.7405  L2_ITS_ICT_Incident_Reportin — Entry into force
  5    2 ITS                       9  0.7401  L2_ITS_ICT_Incident_Reportin — Entry into force


═════════════════════════════════════════════════════════════════
  BM25 RETRIEVER
═════════════════════════════════════════════════════════════

In [14]:
# Cell 11 — Test across all DORA query types
TEST_QUERIES = [
    ("What must be included in the initial notification of a major ICT incident?", "factual_lookup"),
    ("TLPT requirements for significant institutions",                             "exact_term"),
    ("Article 28 general principles ICT third-party risk management",              "article_reference"),
    ("Does DORA apply to crypto-asset service providers?",                         "applicability"),
    ("Is the appointment of an ICT third-party risk function mandatory?",           "obligation_type"),
    ("What do the technical standards say about subcontracting ICT services?",      "cross_level"),
    ("What does DORA say about GDPR fines?",                                       "negative_test"),
]

for query, qtype in TEST_QUERIES:
    results = hybrid_retriever.retrieve(query)
    print(f"[{qtype.upper()}]")
    print(f"Query: {query}")
    show_results(results)

[FACTUAL_LOOKUP]
Query: What must be included in the initial notification of a major ICT incident?
  Results : 5
  #    L Type                    Art   Score  Source / Title
  ──────────────────────────────────────────────────────────────
  1    2 ITS                       9  0.0167  L2_ITS_ICT_Incident_Reportin — Entry into force
  2    3 guideline                 7  0.0167  DORA_ESA_Guidelines_Simplifi — on aggregated reporting.
  3    2 ITS                       9  0.0164  L2_ITS_ICT_Incident_Reportin — Entry into force
  4    2 RTS                       5  0.0164  L2_RTS_ICT_Incident_Reportin — Time limits for the initial 
  5    2 ITS                       9  0.0161  L2_ITS_ICT_Incident_Reportin — Entry into force

[EXACT_TERM]
Query: TLPT requirements for significant institutions
  Results : 5
  #    L Type                    Art   Score  Source / Title
  ──────────────────────────────────────────────────────────────
  1    1 regulation               26  0.0328  L1_DORA_Regulatio

In [15]:
# Cell 12 — Filtered retriever factory
from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
)

def make_filtered_retriever(filters: MetadataFilters, top_k: int = 5):
    return VectorIndexRetriever(
        index=index,
        similarity_top_k=top_k,
        filters=filters,
    )

retriever_level1 = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="level", value=1, operator=FilterOperator.EQ)
]))
retriever_level2 = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="level", value=2, operator=FilterOperator.EQ)
]))
retriever_level3 = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="level", value=3, operator=FilterOperator.EQ)
]))
retriever_rts = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="doc_type", value="RTS", operator=FilterOperator.EQ)
]))
retriever_its = make_filtered_retriever(MetadataFilters(filters=[
    MetadataFilter(key="doc_type", value="ITS", operator=FilterOperator.EQ)
]))

print("✓ Filtered retrievers ready:")
print("  retriever_level1 — Level 1 (regulation + directive)")
print("  retriever_level2 — Level 2 (RTS + ITS)")
print("  retriever_level3 — Level 3 (guidelines + statements)")
print("  retriever_rts    — RTS documents only")
print("  retriever_its    — ITS documents only")

✓ Filtered retrievers ready:
  retriever_level1 — Level 1 (regulation + directive)
  retriever_level2 — Level 2 (RTS + ITS)
  retriever_level3 — Level 3 (guidelines + statements)
  retriever_rts    — RTS documents only
  retriever_its    — ITS documents only


In [16]:
# Cell 13 — Multi-regulation query router
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator
from llama_index.core.retrievers import VectorIndexRetriever, QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever

def make_retriever(level=None, doc_type=None, regulation=None, jurisdiction=None, top_k=8):
    filters = []
    if level        is not None: filters.append(MetadataFilter(key='level',        value=level,        operator=FilterOperator.EQ))
    if doc_type     is not None: filters.append(MetadataFilter(key='doc_type',     value=doc_type,     operator=FilterOperator.EQ))
    if regulation   is not None: filters.append(MetadataFilter(key='regulation',   value=regulation,   operator=FilterOperator.EQ))
    if jurisdiction is not None: filters.append(MetadataFilter(key='jurisdiction', value=jurisdiction, operator=FilterOperator.EQ))
    if filters:
        return VectorIndexRetriever(index=index, similarity_top_k=top_k,
                                    filters=MetadataFilters(filters=filters))
    sem = VectorIndexRetriever(index=index, similarity_top_k=top_k)
    bm  = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k,
                                       stemmer=stemmer, language='english')
    return QueryFusionRetriever(
        retrievers=[sem, bm], llm=Settings.llm, similarity_top_k=top_k,
        num_queries=1, mode='reciprocal_rerank', use_async=False, verbose=False,
    )


def detect_regulation(query):
    q, detected = query.lower(), []
    if any(k in q for k in ['dora', '2022/2554', 'digital operational resilience',
                             'financial entit', 'eba', 'esma', 'eiopa']):
        detected.append('dora')
    if any(k in q for k in ['nis2', 'nis 2', '2022/2555', 'network and information security',
                             'essential entit', 'important entit', 'enisa']):
        detected.append('nis2')
    if any(k in q for k in ['ps21', 'ps21/3', 'ss1/21', 'fg22', 'fca', 'pra',
                             'impact tolerance', 'important business service',
                             'self-assessment', 'operational resilience uk']):
        detected.append('ps21_3')
    return detected


def route_query(query, regulation=None):
    q = query.lower()
    if regulation:
        reg_scope = regulation
    else:
        detected = detect_regulation(query)
        reg_scope = detected[0] if len(detected) == 1 else None

    # NIS2 routing
    if reg_scope == 'nis2':
        if any(k in q for k in ['implementing regulation', 'technical requirement',
                                  'incident notification', '2024/2690']):
            return make_retriever(regulation='nis2', level=2, top_k=10), 'NIS2 Level 2'
        if any(k in q for k in ['enisa', 'guideline', 'security measure']):
            return make_retriever(regulation='nis2', level=3, top_k=8), 'NIS2 Level 3'
        return make_retriever(regulation='nis2', top_k=10), 'NIS2 full corpus'

    # PS21/3 routing
    if reg_scope == 'ps21_3':
        if any(k in q for k in ['ss1/21', 'supervisory statement', 'pra rule']):
            return make_retriever(regulation='ps21_3', doc_type='supervisory_statement', top_k=8), 'PS21/3 Supervisory Statement'
        if any(k in q for k in ['fg22', 'finalised guidance', 'fca guidance']):
            return make_retriever(regulation='ps21_3', doc_type='guidance', top_k=8), 'PS21/3 FCA Guidance'
        if any(k in q for k in ['critical third part', 'ctp', 'dp3/22', 'dp22/3']):
            return make_retriever(regulation='ps21_3', doc_type='discussion_paper', top_k=8), 'PS21/3 CTP'
        return make_retriever(regulation='ps21_3', top_k=10), 'PS21/3 full corpus'

    # DORA routing (all existing Phase 4 rules preserved)
    reg_filter = 'dora' if reg_scope == 'dora' else None
    if any(k in q for k in ['regulatory technical standard', ' rts ', 'rts on',
                             'implementing technical standard', ' its ', 'its on',
                             'technical standard', 'delegated regulation']):
        return make_retriever(level=2, regulation=reg_filter, top_k=12), 'Level 2 (RTS/ITS)'
    if any(k in q for k in ['tlpt', 'penetration test', 'threat-led', 'tiber']):
        return make_retriever(regulation=reg_filter, top_k=12), 'Full corpus (TLPT)'
    if any(k in q for k in ['subcontract', 'chain', 'group entit', 'concentrat', 'third-party provider']):
        return make_retriever(regulation=reg_filter, top_k=12), 'Full corpus (adversarial)'
    if any(k in q for k in ['guideline', 'aggregate', 'oversight cooperation', 'public statement']):
        return make_retriever(level=3, regulation=reg_filter, top_k=8), 'Level 3 (guidelines)'
    if any(k in q for k in ['template', 'reporting form']):
        return make_retriever(doc_type='ITS', regulation=reg_filter, top_k=8), 'ITS only'
    if any(k in q for k in ['article ', 'regulation ', 'directive ', ' shall ', ' must ', '2022/2554']):
        return make_retriever(level=1, regulation=reg_filter, top_k=8), 'Level 1 (main regulation)'
    if reg_filter:
        return make_retriever(regulation=reg_filter, top_k=8), f'{reg_filter.upper()} full corpus'
    return make_retriever(top_k=10), 'Full corpus (all regulations)'


def retrieve(query, regulation=None, top_k=None, verbose=False):
    retriever, scope = route_query(query, regulation=regulation)
    results = retriever.retrieve(query)
    if verbose:
        print(f'Scope   : {scope}')
        show_results(results)
    return results, scope


def retrieve_regulation(query, regulation, top_k=8):
    return retrieve(query, regulation=regulation, top_k=top_k)


print('✓ Multi-regulation query router ready')
print('  make_retriever() now supports regulation + jurisdiction filters')
print('  retrieve(query, regulation="nis2") — scoped to one regulation')
print('  retrieve_regulation(query, reg)    — convenience alias')
print('  Existing retrieve(query) unchanged — DORA behaviour preserved')


✓ Multi-regulation query router ready
  make_retriever() now supports regulation + jurisdiction filters
  retrieve(query, regulation="nis2") — scoped to one regulation
  retrieve_regulation(query, reg)    — convenience alias
  Existing retrieve(query) unchanged — DORA behaviour preserved


In [17]:
# Cell 14 — HyDE: Hypothetical Document Embedding (multi-regulation)
# Reference: Gao et al. (2022)

HYDE_PROMPT_TEMPLATE = ('You are an expert in regulatory law covering three frameworks:\n'
    '- EU DORA (Digital Operational Resilience Act 2022/2554)\n'
    '- EU NIS2 (Network and Information Security Directive 2022/2555)\n'
    '- UK PS21/3 (FCA/PRA Operational Resilience Policy Statement)\n\n'
    'Write a short passage (4-6 sentences) that could plausibly appear in whichever '
    'of these regulations is most relevant to the question below. Write as actual '
    'regulatory text using SHALL/SHOULD/MAY, citing Articles or paragraph references '
    'where appropriate. Do NOT answer conversationally.\n\n'
    'Question: {question}\n\nHypothetical regulatory passage:')


def llm_complete_with_retry(prompt, max_retries=4):
    for attempt in range(1, max_retries + 1):
        try:
            return llm.complete(prompt).text.strip()
        except Exception as e:
            err = str(e)
            is_rate = any(k in err for k in ['rate_limit_exceeded', 'RateLimitError',
                                              'tokens per', 'requests per', '429', 'retry in'])
            if not is_rate: raise
            if attempt == max_retries:
                raise RuntimeError('Rate limit not resolved after retries.') from e
            match = _re.search(r'try again in (\d+(?:\.\d+)?)s', err, _re.IGNORECASE)
            wait  = float(match.group(1)) + 2 if match else 30
            print(f'    Groq rate limit ({attempt}/{max_retries}). Waiting {wait:.0f}s...')
            time.sleep(wait)


def generate_hypothesis(question):
    return llm_complete_with_retry(HYDE_PROMPT_TEMPLATE.format(question=question))


def retrieve_with_hyde(query, regulation=None, verbose=False):
    hypothesis       = generate_hypothesis(query)
    retriever, scope = route_query(query, regulation=regulation)
    nodes            = retriever.retrieve(hypothesis)
    if verbose:
        print(f'Query     : {query}')
        print(f'Regulation: {regulation or "auto-detect"}')
        print(f'Scope     : {scope}')
        print(f'Hypothesis: {hypothesis[:200]}...')
        show_results(nodes)
    return nodes, scope, hypothesis


print('✓ HyDE ready (multi-regulation — DORA + NIS2 + PS21/3)')
print('  retrieve_with_hyde(query)              — auto-detect regulation')
print('  retrieve_with_hyde(query, reg="nis2")   — scoped to one regulation')


✓ HyDE ready (multi-regulation — DORA + NIS2 + PS21/3)
  retrieve_with_hyde(query)              — auto-detect regulation
  retrieve_with_hyde(query, reg="nis2")   — scoped to one regulation


In [18]:
# Cell 15 — Query router tests (multi-regulation)
routing_tests = [
    ('What does the RTS on ICT risk management say about governance?',        'dora'),
    ('TLPT frequency and scoping requirements',                                'dora'),
    ('Article 19 requirements for ICT incident classification',               'dora'),
    ('What are the NIS2 incident reporting obligations for essential entities?','nis2'),
    ('NIS2 implementing regulation technical cybersecurity requirements',       'nis2'),
    ('What is an impact tolerance under PS21/3?',                              'ps21_3'),
    ('PRA supervisory statement SS1/21 operational resilience',                'ps21_3'),
    ('How do incident reporting requirements compare across regulations?',      None),
]

print('QUERY ROUTER TESTS — MULTI-REGULATION')
print('=' * 72)
print(f"  {'Query':<52} {'Detected':<12} Scope")
print('  ' + '-' * 72)
for query, expected in routing_tests:
    detected = detect_regulation(query)
    _, scope = route_query(query)
    det_str  = ','.join(detected) if detected else 'general'
    ok = '✓' if (not expected or expected in detected) else '?'
    print(f'  {ok} {query[:52]:<52} {det_str:<12} {scope}')


QUERY ROUTER TESTS — MULTI-REGULATION
  Query                                                Detected     Scope
  ------------------------------------------------------------------------
  ? What does the RTS on ICT risk management say about g general      Level 2 (RTS/ITS)
  ? TLPT frequency and scoping requirements              general      Full corpus (TLPT)
  ? Article 19 requirements for ICT incident classificat general      Level 1 (main regulation)
  ✓ What are the NIS2 incident reporting obligations for nis2         NIS2 full corpus
  ✓ NIS2 implementing regulation technical cybersecurity nis2         NIS2 Level 2
  ✓ What is an impact tolerance under PS21/3?            ps21_3       PS21/3 full corpus
  ✓ PRA supervisory statement SS1/21 operational resilie ps21_3       PS21/3 Supervisory Statement
  ✓ How do incident reporting requirements compare acros general      Full corpus (all regulations)


In [ ]:
# Cell 16 — Retrieval evaluation: DORA baseline + NIS2 + PS21/3 spot checks

# Part A: DORA baseline (same 10 queries as Phase 4a — confirms no regression)
print('PART A — DORA BASELINE (regression check vs Phase 4)')
print('=' * 65)
EVAL_SET = [
    ('Timelines for major ICT incident reporting',                 'L1_DORA_Regulation', 19),
    ('Management body responsibilities for ICT risk',              'L1_DORA_Regulation',  5),
    ('General principles ICT third-party risk management',         'L1_DORA_Regulation', 28),
    ('Contractual provisions ICT third-party service providers',   'L1_DORA_Regulation', 30),
    ('Business continuity and disaster recovery plans',            'L1_DORA_Regulation', 11),
    ('ICT incident classification criteria thresholds',            'L2_RTS_ICT_Incident_Class', None),
    ('Threat-led penetration testing TLPT scope',                  'L2_RTS_TLPT', 1),
    ('ICT third-party subcontracting chain requirements',          'L2_RTS_Subcontracting', None),
    ('Register of information ICT third-party providers',          'L2_ITS_Register_of_Information', None),
    ('Estimating aggregate annual costs ICT incidents guidelines', 'L3_Guidelines_Costs_and_Losses', None),
]
source_hits = art_hits = full_hits = 0
total = len(EVAL_SET)
print(f"\n  {'#':<3} {'Src':>4} {'Art':>4}  Query")
print('  ' + '-' * 60)
for i, (query, exp_src, exp_art) in enumerate(EVAL_SET, 1):
    results, _ = retrieve(query, regulation='dora')
    srcs = [n.metadata.get('source_doc','') for n in results]
    arts = [n.metadata.get('article_num',-1) for n in results]
    sh = any(exp_src in s for s in srcs)
    ah = (exp_art is None) or (exp_art in arts)
    if sh: source_hits += 1
    if ah: art_hits    += 1
    if sh and ah: full_hits += 1
    print(f"  {i:<3} {'✓' if sh else '✗':>4} {'✓' if ah else '✗':>4}  {query[:50]}")
print('  ' + '-' * 60)
print(f'  Full hit: {full_hits}/{total} ({100*full_hits//total}%)')
print(f'  {"✓ No regression" if full_hits/total >= 0.7 else "⚠ Regression detected"}')

# Part B: NIS2 spot checks
print('\n\nPART B — NIS2 SPOT CHECKS')
print('=' * 65)
nis2_checks = [
    ('cybersecurity risk management obligations essential entities', 'NIS2'),
    ('incident reporting timelines NIS2 Article 23',               'NIS2'),
    ('technical cybersecurity requirements implementing regulation', 'NIS2_IR'),
]
for query, exp_src in nis2_checks:
    results, scope = retrieve(query, regulation='nis2')
    hit = any(exp_src in n.metadata.get('source_doc','') for n in results)
    print(f"  {'✓' if hit else '✗'} [{scope}] {query[:50]}")
    for n in results[:2]:
        print(f"      {n.metadata.get('source_doc','')[:45]}")

# Part C: PS21/3 spot checks
print('\n\nPART C — PS21/3 SPOT CHECKS')
print('=' * 65)
ps213_checks = [
    ('impact tolerance important business service self-assessment', 'PS21_3'),
    ('PRA supervisory statement operational resilience',            'SS1_21'),
    ('FCA finalised guidance FCA FG22',                            'FCA_FG22'),
]
for query, exp_src in ps213_checks:
    results, scope = retrieve(query, regulation='ps21_3')
    hit = any(exp_src in n.metadata.get('source_doc','') for n in results)
    print(f"  {'✓' if hit else '✗'} [{scope}] {query[:50]}")
    for n in results[:2]:
        print(f"      {n.metadata.get('source_doc','')[:45]}")


PART A — DORA BASELINE (regression check vs Phase 4)

  #    Src  Art  Query
  ------------------------------------------------------------
  1      ✓    ✗  Timelines for major ICT incident reporting
  2      ✓    ✓  Management body responsibilities for ICT risk
  3      ✓    ✓  General principles ICT third-party risk management
  4      ✓    ✓  Contractual provisions ICT third-party service pro
  5      ✓    ✓  Business continuity and disaster recovery plans
  6      ✓    ✓  ICT incident classification criteria thresholds
  7      ✗    ✓  Threat-led penetration testing TLPT scope
  8      ✓    ✓  ICT third-party subcontracting chain requirements
  9      ✓    ✓  Register of information ICT third-party providers
  10     ✓    ✓  Estimating aggregate annual costs ICT incidents gu
  ------------------------------------------------------------
  Full hit: 8/10 (80%)
  ✓ No regression


PART B — NIS2 SPOT CHECKS
  ✓ [NIS2 full corpus] cybersecurity risk management obligations essentia
    

In [20]:
# Cell 15 — HyDE demonstration: standard vs HyDE retrieval
#
# Compares standard retrieval against HyDE on 3 representative queries.
# Shows the generated hypothesis and how it changes which chunks are retrieved.
# Uses 1 Groq 70b call per query (3 calls total).

HYDE_DEMO_QUERIES = [
    # Adversarial — short query, answer buried in penalty articles
    ("What happens if a financial entity fails to report an ICT incident on time?",
     "adversarial"),
    # Vague / general — would benefit from hypothesis expansion
    ("How should financial entities manage their exposure to third-party ICT providers?",
     "general"),
    # Cross-level — needs both L1 and L2 content
    ("What are the detailed criteria for classifying an ICT incident as major?",
     "cross_level"),
]

print("HyDE DEMONSTRATION — Standard vs HyDE Retrieval")
print("=" * 65)
print("(Uses 3 Groq 70b calls — ~10s total)\n")

hyde_improvements = []

for query, qtype in HYDE_DEMO_QUERIES:
    print(f"{'─'*65}")
    print(f"[{qtype.upper()}] {query}")

    # Standard retrieval — embed the raw question
    std_nodes, std_scope = retrieve(query)
    std_srcs = [
        (n.metadata.get('article_num', '?'), n.metadata.get('source_doc', '')[:30])
        for n in std_nodes[:5]
    ]

    # HyDE retrieval — embed a hypothetical regulatory passage instead
    hyde_nodes, hyde_scope, hypothesis = retrieve_with_hyde(query)
    hyde_srcs = [
        (n.metadata.get('article_num', '?'), n.metadata.get('source_doc', '')[:30])
        for n in hyde_nodes[:5]
    ]

    print(f"\nHypothesis (first 250 chars):")
    print(f"  {hypothesis[:250]}...")

    print(f"\nStandard top-5  (embedding: raw question):")
    for art, src in std_srcs:
        print(f"  Art {str(art):<4}  {src}")

    print(f"\nHyDE top-5  (embedding: hypothetical passage):")
    for art, src in hyde_srcs:
        print(f"  Art {str(art):<4}  {src}")

    # Count how many unique chunks HyDE found that standard missed
    std_set  = {n.id_ for n in std_nodes[:5]}
    hyde_set = {n.id_ for n in hyde_nodes[:5]}
    new_chunks = hyde_set - std_set
    print(f"\nNew chunks found by HyDE (not in standard top-5): {len(new_chunks)}")
    hyde_improvements.append(len(new_chunks))

    time.sleep(3)   # polite delay between Groq calls

print(f"\n{'='*65}")
print(f"Average new chunks per query: {sum(hyde_improvements)/len(hyde_improvements):.1f}")
print("Higher = HyDE found more unique results vs standard retrieval")
print("\nConclusion: HyDE improves retrieval by embedding a hypothesis that")
print("looks like the target document rather than the short raw question.")

HyDE DEMONSTRATION — Standard vs HyDE Retrieval
(Uses 3 Groq 70b calls — ~10s total)

─────────────────────────────────────────────────────────────────
[ADVERSARIAL] What happens if a financial entity fails to report an ICT incident on time?

Hypothesis (first 250 chars):
  In the event of an ICT incident, financial entities SHALL report the incident to the relevant competent authority within the timeframe specified in Article 24 of this Regulation. Failure to report an ICT incident in a timely manner MAY result in addi...

Standard top-5  (embedding: raw question):
  Art 17    L1_DORA_Regulation_2022_2554
  Art 23    L2_RTS_ICT_Risk_Management
  Art 9     L2_ITS_ICT_Incident_Reporting
  Art 9     L2_ITS_ICT_Incident_Reporting
  Art 9     L2_ITS_ICT_Incident_Reporting

HyDE top-5  (embedding: hypothetical passage):
  Art 6     L2_ITS_ICT_Incident_Reporting
  Art 17    L1_DORA_Regulation_2022_2554
  Art 9     L2_ITS_ICT_Incident_Reporting
  Art 7     L2_ITS_ICT_Incident_Reporting
  Art 

In [ ]:
# Cell 16 — Test query router
routing_tests = [
    "What does the RTS on ICT risk management say about governance?",
    "TLPT frequency and scoping requirements",
    "What are the incident reporting templates?",
    "Guidelines on estimating aggregate costs of ICT incidents",
    "Article 19 requirements for ICT incident classification",
    "Does DORA apply to investment firms?",
]

print("QUERY ROUTER TESTS")
print("=" * 65)
for query in routing_tests:
    _, scope = route_query(query)
    print(f"  Query : {query[:60]}")
    print(f"  Routed: {scope}\n")

QUERY ROUTER TESTS
  Query : What does the RTS on ICT risk management say about governanc
  Routed: Level 2 (RTS/ITS)

  Query : TLPT frequency and scoping requirements
  Routed: Full corpus (TLPT)

  Query : What are the incident reporting templates?
  Routed: ITS only

  Query : Guidelines on estimating aggregate costs of ICT incidents
  Routed: Level 3 (guidelines)

  Query : Article 19 requirements for ICT incident classification
  Routed: Level 1 (main regulation)

  Query : Does DORA apply to investment firms?
  Routed: DORA full corpus



In [ ]:
# Cell 17 — Retrieval evaluation (hit rate)
EVAL_SET = [
    ("Timelines for major ICT incident reporting",                  "L1_DORA_Regulation",              19),
    ("Management body responsibilities for ICT risk",               "L1_DORA_Regulation",               5),
    ("General principles ICT third-party risk management",          "L1_DORA_Regulation",              28),
    ("Contractual provisions ICT third-party service providers",    "L1_DORA_Regulation",              30),
    ("Business continuity and disaster recovery plans",             "L1_DORA_Regulation",              11),
    ("ICT incident classification criteria thresholds",             "L2_RTS_ICT_Incident_Class",        None),
    ("Threat-led penetration testing TLPT scope",                   "L2_RTS_TLPT",                      1),
    ("ICT third-party subcontracting chain requirements",           "L2_RTS_Subcontracting",            None),
    ("Register of information ICT third-party providers",           "L2_ITS_Register_of_Information",   None),
    ("Estimating aggregate annual costs ICT incidents guidelines",  "L3_Guidelines_Costs_and_Losses", None),
]

source_hits = 0
art_hits    = 0
full_hits   = 0
total       = len(EVAL_SET)

print(f"Retrieval evaluation ({total} queries)\n")
print(f"  {'#':<3} {'Src':>4} {'Art':>4}  Query")
print("  " + "─" * 60)

for i, (query, exp_src, exp_art) in enumerate(EVAL_SET, 1):
    results, _ = retrieve(query, top_k=8, verbose=False)
    retrieved_srcs = [n.metadata.get("source_doc", "") for n in results]
    retrieved_arts = [n.metadata.get("article_num", -1) for n in results]

    src_hit = any(exp_src in s for s in retrieved_srcs)
    art_hit = (exp_art is None) or (exp_art in retrieved_arts)

    if src_hit: source_hits += 1
    if art_hit: art_hits    += 1
    if src_hit and art_hit: full_hits += 1

    print(f"  {i:<3} {'✓' if src_hit else '✗':>4} {'✓' if art_hit else '✗':>4}  {query[:50]}")

print("  " + "─" * 60)
print(f"\n  Source hit rate  : {source_hits}/{total} ({100*source_hits//total}%)")
print(f"  Article hit rate : {art_hits}/{total} ({100*art_hits//total}%)")
print(f"  Full hit rate    : {full_hits}/{total} ({100*full_hits//total}%)")
print(f"\n  Baseline target: >70% full hit rate")
status = "✓ READY for Phase 5" if full_hits/total >= 0.7 else "⚠ Consider tuning top_k or chunk size"
print(f"  {status}")

Retrieval evaluation (10 queries)

  #    Src  Art  Query
  ────────────────────────────────────────────────────────────
  1      ✓    ✗  Timelines for major ICT incident reporting
  2      ✓    ✓  Management body responsibilities for ICT risk
  3      ✓    ✓  General principles ICT third-party risk management
  4      ✓    ✓  Contractual provisions ICT third-party service pro
  5      ✓    ✓  Business continuity and disaster recovery plans
  6      ✓    ✓  ICT incident classification criteria thresholds
  7      ✗    ✗  Threat-led penetration testing TLPT scope
  8      ✓    ✓  ICT third-party subcontracting chain requirements
  9      ✓    ✓  Register of information ICT third-party providers
  10     ✓    ✓  Estimating aggregate annual costs ICT incidents gu
  ────────────────────────────────────────────────────────────

  Source hit rate  : 9/10 (90%)
  Article hit rate : 8/10 (80%)
  Full hit rate    : 8/10 (80%)

  Baseline target: >70% full hit rate
  ✓ READY for Phase 5


In [ ]:
# Cell 18 — Hit rate evaluation: NIS2 + PS21/3
#
# Measures source hit rate and full hit rate for NIS2 and PS21/3 corpora.
# article_num is None for most PS21/3 chunks (section_ref used instead)
# so we only measure source hit rate for UK documents.

print('NIS2 HIT RATE EVALUATION')
print('=' * 65)

NIS2_EVAL = [
    ('Incident reporting obligations for essential entities',         'NIS2_Directive_2022_2555',        23),
    ('Cybersecurity risk management measures Article 21',             'NIS2_Directive_2022_2555',        21),
    ('Scope of NIS2 which entities are covered',                      'NIS2_Directive_2022_2555',         3),
    ('Significant incidents notification competent authority',        'NIS2_Directive_2022_2555',        23),
    ('Technical and methodological requirements cybersecurity',       'NIS2_IR_2024_2690',             None),
    ('Incident notification format technical requirements',           'NIS2_IR_2024_2690',             None),
    ('Critical entities resilience obligations CER Directive',        'NIS2_CER_Directive_2022_2557',  None),
    ('Security measures supply chain risk management NIS2',           'NIS2_Directive_2022_2555',        21),
]

nis2_src = nis2_full = 0
print(f"\n  {'#':<3} {'Src':>4} {'Art':>4}  Query")
print('  ' + '─' * 62)
for i, (query, exp_src, exp_art) in enumerate(NIS2_EVAL, 1):
    results, _ = retrieve(query, regulation='nis2')
    srcs = [n.metadata.get('source_doc', '') for n in results]
    arts = [n.metadata.get('article_num', -1) for n in results]
    sh = any(exp_src in s for s in srcs)
    ah = (exp_art is None) or (exp_art in arts)
    if sh: nis2_src  += 1
    if sh and ah: nis2_full += 1
    print(f"  {i:<3} {'✓' if sh else '✗':>4} {'✓' if ah else '─':>4}  {query[:52]}")

n2 = len(NIS2_EVAL)
print('  ' + '─' * 62)
print(f'  Source hit : {nis2_src}/{n2} ({100*nis2_src//n2}%)')
print(f'  Full hit   : {nis2_full}/{n2} ({100*nis2_full//n2}%)')
print(f'  (─ = article_num not expected for this doc)')

print('\n\nPS21/3 HIT RATE EVALUATION')
print('=' * 65)
print('Note: PS21/3 uses paragraph/chapter refs not Article numbers.')
print('      Only source hit rate is measured.\n')

PS213_EVAL = [
    ('Impact tolerance setting important business services',  'PS21_3_FCA_PRA'),
    ('Self-assessment operational resilience boards',         'SS1_21_PRA'),
    ('Mapping and testing important business services',       'PS21_3_FCA_PRA'),
    ('Scenario testing severe but plausible disruptions',     'SS1_21_PRA'),
    ('FCA guidance staying within impact tolerances',         'FCA_FG22_5'),
    ('Critical third parties systemic risk UK finance',       'DP3_22_Critical'),
    ('Implementation timeline operational resilience March',  'PS21_3_FCA_PRA'),
    ('Governance accountability operational resilience',      'SS1_21_PRA'),
]

ps213_src = 0
print(f"  {'#':<3} {'Src':>4}  Query")
print('  ' + '─' * 58)
for i, (query, exp_src) in enumerate(PS213_EVAL, 1):
    results, _ = retrieve(query, regulation='ps21_3')
    srcs = [n.metadata.get('source_doc', '') for n in results]
    sh = any(exp_src in s for s in srcs)
    if sh: ps213_src += 1
    print(f"  {i:<3} {'✓' if sh else '✗':>4}  {query[:52]}")

n3 = len(PS213_EVAL)
print('  ' + '─' * 58)
print(f'  Source hit : {ps213_src}/{n3} ({100*ps213_src//n3}%)')

print('\n\nCOMPLETE RETRIEVAL SUMMARY')
print('=' * 65)
print(f"  {'Corpus':<20} {'Source Hit':>10} {'Full Hit':>10}")
print('  ' + '─' * 42)
print(f"  {'DORA':<20} {source_hits}/{total:>8} ({100*source_hits//total}%)  {full_hits}/{total} ({100*full_hits//total}%)")
print(f"  {'NIS2':<20} {nis2_src}/{n2:>8} ({100*nis2_src//n2}%)  {nis2_full}/{n2} ({100*nis2_full//n2}%)")
print(f"  {'PS21/3':<20} {ps213_src}/{n3:>8} ({100*ps213_src//n3}%)  n/a (no article nums)")

NIS2 HIT RATE EVALUATION

  #    Src  Art  Query
  ──────────────────────────────────────────────────────────────
  1      ✓    ✓  Incident reporting obligations for essential entitie
  2      ✓    ✓  Cybersecurity risk management measures Article 21
  3      ✓    ✓  Scope of NIS2 which entities are covered
  4      ✓    ✓  Significant incidents notification competent authori
  5      ✓    ✓  Technical and methodological requirements cybersecur
  6      ✓    ✓  Incident notification format technical requirements
  7      ✓    ✓  Critical entities resilience obligations CER Directi
  8      ✗    ─  Security measures supply chain risk management NIS2
  ──────────────────────────────────────────────────────────────
  Source hit : 7/8 (87%)
  Full hit   : 7/8 (87%)
  (─ = article_num not expected for this doc)


PS21/3 HIT RATE EVALUATION
Note: PS21/3 uses paragraph/chapter refs not Article numbers.
      Only source hit rate is measured.

  #    Src  Query
  ──────────────────────────────

In [26]:
# Cell 19 — Save retrieval config for Phase 5b
retrieval_config = {
    'created_at':            datetime.datetime.now().isoformat(),
    'llm':                   'groq/llama-3.3-70b-versatile',
    'hybrid_top_k':          8,
    'fusion_mode':           'reciprocal_rerank',
    'query_router':          'keyword_based_multi_regulation',
    'regulations_supported': ['dora', 'nis2', 'ps21_3', 'cross_reg'],
    'retrievers_available':  ['semantic', 'bm25', 'hybrid', 'regulation_filtered',
                               'jurisdiction_filtered', 'level_filtered', 'doc_type_filtered'],
    'default_retriever':     'hybrid',
    'eval_hit_rate_dora':    f'{full_hits}/{total} ({100*full_hits//total}%)',
    'notes': ('Multi-regulation pipeline. retrieve(query, regulation="nis2") scopes to one '
              'regulation. retrieve_regulation() is convenience alias. HyDE covers DORA+NIS2+PS21/3.')
}
path = BASE_DIR / 'retrieval_config.json'
path.write_text(json.dumps(retrieval_config, indent=2))
print(f'✓ Retrieval config saved: {path}')
print(json.dumps(retrieval_config, indent=2))


✓ Retrieval config saved: dora_rag/retrieval_config.json
{
  "created_at": "2026-04-19T19:51:21.999452",
  "llm": "groq/llama-3.3-70b-versatile",
  "hybrid_top_k": 8,
  "fusion_mode": "reciprocal_rerank",
  "query_router": "keyword_based_multi_regulation",
  "regulations_supported": [
    "dora",
    "nis2",
    "ps21_3",
    "cross_reg"
  ],
  "retrievers_available": [
    "semantic",
    "bm25",
    "hybrid",
    "regulation_filtered",
    "jurisdiction_filtered",
    "level_filtered",
    "doc_type_filtered"
  ],
  "default_retriever": "hybrid",
  "eval_hit_rate_dora": "8/10 (80%)",
  "notes": "Multi-regulation pipeline. retrieve(query, regulation=\"nis2\") scopes to one regulation. retrieve_regulation() is convenience alias. HyDE covers DORA+NIS2+PS21/3."
}


In [28]:
# Cell 20 — Phase 4b summary
dora_cnt  = sum(1 for c in raw_chunks if c.get('regulation')=='dora')
nis2_cnt  = sum(1 for c in raw_chunks if c.get('regulation')=='nis2')
ps213_cnt = sum(1 for c in raw_chunks if c.get('regulation')=='ps21_3')
xreg_cnt  = sum(1 for c in raw_chunks if c.get('regulation')=='cross_reg')

print('\n' + '=' * 65)
print('PHASE 4b COMPLETE — MULTI-REGULATION RETRIEVAL PIPELINE')
print('=' * 65)
print(f"""
Corpus           : {len(raw_chunks)} chunks
  DORA (EU)      : {dora_cnt}
  NIS2 (EU)      : {nis2_cnt}
  PS21/3 (UK)    : {ps213_cnt}
  Cross-reg      : {xreg_cnt}

Retrieval modes:
  retrieve(query)                      auto-detect regulation
  retrieve(query, regulation='dora')   DORA only
  retrieve(query, regulation='nis2')   NIS2 only
  retrieve(query, regulation='ps21_3') PS21/3 only
  retrieve_regulation(query, reg)      convenience alias
  retrieve_with_hyde(query, reg)       HyDE, scoped

Advanced techniques:
  Hybrid Search (#4) BM25 + vector, RRF fusion
  HyDE (#2)          Gao et al. 2022, multi-regulation aware

RETRIEVAL HIT RATES:
  {'Corpus':<20} {'Source Hit':>12} {'Full Hit':>10}
  {'─'*44}
  {'DORA':<20} {source_hits:>3}/{total:<3} ({100*source_hits//total}%)  {full_hits}/{total} ({100*full_hits//total}%)
  {'NIS2':<20} {nis2_src:>3}/{n2:<3} ({100*nis2_src//n2}%)  {nis2_full}/{n2} ({100*nis2_full//n2}%)
  {'PS21/3':<20} {ps213_src:>3}/{n3:<3} ({100*ps213_src//n3}%)  n/a (section refs, no article nums)

Targets: Source hit ≥80%  Full hit ≥70%
  DORA   : {'✓' if source_hits/total >= 0.8 else '⚠'}  {'✓' if full_hits/total >= 0.7 else '⚠'}
  NIS2   : {'✓' if nis2_src/n2 >= 0.8 else '⚠'}  {'✓' if nis2_full/n2 >= 0.7 else '⚠'}
  PS21/3 : {'✓' if ps213_src/n3 >= 0.8 else '⚠'}  n/a

Next step -> 05b_generation_multiregulation.ipynb
""")


PHASE 4b COMPLETE — MULTI-REGULATION RETRIEVAL PIPELINE

Corpus           : 826 chunks
  DORA (EU)      : 390
  NIS2 (EU)      : 189
  PS21/3 (UK)    : 119
  Cross-reg      : 128

Retrieval modes:
  retrieve(query)                      auto-detect regulation
  retrieve(query, regulation='dora')   DORA only
  retrieve(query, regulation='nis2')   NIS2 only
  retrieve(query, regulation='ps21_3') PS21/3 only
  retrieve_regulation(query, reg)      convenience alias
  retrieve_with_hyde(query, reg)       HyDE, scoped

Advanced techniques:
  Hybrid Search (#4) BM25 + vector, RRF fusion
  HyDE (#2)          Gao et al. 2022, multi-regulation aware

RETRIEVAL HIT RATES:
  Corpus                 Source Hit   Full Hit
  ────────────────────────────────────────────
  DORA                   9/10  (90%)  8/10 (80%)
  NIS2                   7/8   (87%)  7/8 (87%)
  PS21/3                 8/8   (100%)  n/a (section refs, no article nums)

Targets: Source hit ≥80%  Full hit ≥70%
  DORA   : ✓  ✓
  NIS2 